# Markov Chain Baseline
Builds a first-order Markov transition model from training piano rolls,
generates 5 sample MIDIs, visualizes them, and reports evaluation metrics.

Run from the project root, or set the working directory to the root before executing.

In [ ]:
import sys
import os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
from config import SPLIT_DIR, OUTPUTS_DIR, PLOTS_DIR
from evaluation.baseline_markov import build_markov_model, markov_generate
from evaluation.metrics import evaluate_all, print_metrics_table
from generation.midi_export import piano_roll_to_midi

os.makedirs(os.path.join(OUTPUTS_DIR, 'generated_midis', 'baselines'), exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Load training data
print("Loading training data...")
X_train = np.load(os.path.join(SPLIT_DIR, 'train_data.npy'))
y_train = np.load(os.path.join(SPLIT_DIR, 'train_labels.npy'))
print(f"  Loaded {len(X_train)} training segments.")

In [ ]:
# Build Markov model
print("Building Markov transition model...")
transitions = build_markov_model(X_train)
print(f"  Unique states: {len(transitions)}")

In [ ]:
# Generate 5 Markov samples
print("Generating 5 Markov samples...")
markov_rolls = []
for i in range(1, 6):
    seed = X_train[np.random.randint(len(X_train))][0]
    roll = markov_generate(transitions, seed)
    markov_rolls.append(roll)
    path = os.path.join(OUTPUTS_DIR, 'generated_midis', 'baselines', f'markov_{i}.mid')
    piano_roll_to_midi(roll, save_path=path)
    print(f"  markov_{i}.mid saved")

In [ ]:
# Visualize generated piano rolls
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(markov_rolls[i].T, aspect='auto', origin='lower',
              cmap='Blues', interpolation='nearest')
    ax.set_title(f"Markov Sample {i+1}")
    ax.set_xlabel("Time Steps")
    ax.set_ylabel("Pitch")
plt.suptitle("Markov Chain Baseline - Generated Piano Rolls")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'markov_samples.png'), dpi=150)
plt.show()

In [ ]:
# Evaluation metrics
metrics_list = [evaluate_all(r) for r in markov_rolls]
avg_rhythm   = float(np.mean([m['rhythm_diversity'] for m in metrics_list]))
avg_rep      = float(np.mean([m['repetition_ratio'] for m in metrics_list]))
print(f"\nMarkov Baseline Metrics (avg over 5 samples):")
print(f"  Rhythm Diversity: {avg_rhythm:.3f}")
print(f"  Repetition Ratio: {avg_rep:.3f}")